In [6]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ['GROQ_API_KEY'] = os.getenv("GROQ_API_KEY")
os.environ["HUGGINGFACE_API_KEY"] = os.getenv("HUGGINGFACE_API_KEY")
os.environ["TAVILY_API_KEY"] = os.getenv("TAVILY_API_KEY")

from langchain_classic.text_splitter import RecursiveCharacterTextSplitter
from langchain_groq import ChatGroq
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings

#SET EMBEDDINGS
embeddings = HuggingFaceEmbeddings()


In [7]:
#DOCS TO INDEX
urls = [
    "https://lilianweng.github.io/posts/2023-06-23-agent/",
    "https://lilianweng.github.io/posts/2023-10-25-adv-attack-llm/",
    "https://lilianweng.github.io/posts/2023-03-15-prompt-engineering/"
]

#LOAD
docs = [WebBaseLoader(url).load() for url in urls]
docs_list = [item for sublist in docs for item in sublist]

#SPLIT
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=500,
    chunk_overlap=50
)

docs_splits = text_splitter.split_documents(docs_list)

#ADD TO VECTORSORE DB

vectorstore = FAISS.from_documents(
    documents=docs_splits,
    embedding=embeddings
)

#Retriver
retriever = vectorstore.as_retriever()

In [9]:
#ROUTER
from typing import Literal
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field

#DATA MODEL
class RouteQuery(BaseModel):
    """Routes a user query to the most relevant documents"""

    datasource : Literal['vectorstore','web_search'] = Field(description="Given a user question choose to route it to web search or vector store")

## LLM with Function Call
llm = ChatGroq(model="qwen/qwen3-32b")

structured_llm_router = llm.with_structured_output(RouteQuery)


In [10]:
#PROMPT
system = """You are an expert at routing the user question to a vectorstore or websearch. 
            The vectorstore contains documents related to agents, prompt engineering, and adversial attacks. Use the vector store for questions on these topucs, otherwise choose web search
"""

route_prompt = ChatPromptTemplate.from_messages(
    [
        ("system",system),
        ("human",'{question}')
    ]
)

question_route = route_prompt | structured_llm_router

In [11]:
print(question_route.invoke({'question':"Who won the cricket world cup 2026?"}))

datasource='web_search'


In [12]:
print(question_route.invoke({'question':"What are the types of agent memory?"}))

datasource='vectorstore'


In [14]:
##Retrival Grader
#Data Model
class GradeDocuments(BaseModel):
    "Binary Score for relevance check on retrieved documents"

    binary_score: str = Field(description="Documents are relevant to the question, 'yes' or 'no'")

##LLM with Function Call
llm = ChatGroq(model="qwen/qwen3-32b")

structured_llm_grader = llm.with_structured_output(GradeDocuments)

#Prompt
system = """You are a grader assessing relevance of a retrieved documents to a user question.\n
            if the document contains keyword(s) or semantic meaning related to the user question, grade it as relevant.\n
            if does not need to be stringent test. the goal is to filter out errornous retrievals.\n Give a binary score 'yes' or 'no' to indicate whether the documents retrieved is relevant to the question."""

grader_prompt = ChatPromptTemplate.from_messages(
    [
        ('system',system),
        ('human',"Retrieved Documents: \n\n {documents}\n\n User Question: \n{question}")
    ]
)

retrieval_grader = grader_prompt | structured_llm_grader

question = 'Agent Memory'

docs = retriever.invoke(question)
docs_text = docs[1].page_content

print(retrieval_grader.invoke({
    "question":question,
    "documents":docs_text
}))

binary_score='yes'


In [17]:
#Generate
from langchain_classic import hub
from langchain_core.output_parsers import StrOutputParser

#Prompt
prompt = hub.pull('rlm/rag-prompt')

#LLM
llm = ChatGroq(model="qwen/qwen3-32b")

#Post Processing
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

#chain
rag_chain = prompt | llm | StrOutputParser()

#Run
generation = rag_chain.invoke({
    'context':docs,
    'question':question
})

print(generation)


<think>
Okay, so the user is asking about "Agent Memory" in the context of LLM-powered autonomous agents. Let me look through the provided documents to find relevant information.

The first document mentions that in a LLM-powered agent system, memory is a key component. It differentiates between short-term and long-term memory. Short-term memory is related to in-context learning, like prompt engineering, where the model uses the immediate context. Long-term memory allows the agent to retain information over time using external vector stores and fast retrieval. 

The second document talks about Reflexion, a framework that uses dynamic memory and self-reflection. It uses a working memory to store reflections, which guide future actions. There's also mention of challenges like finite context length affecting memory, where external storage helps but isn't as effective as full attention.

The third document discusses challenges such as the limited context length of LLMs, which restricts his

In [22]:
##HALLUCINATION GRADER

#Data Model
class hallucination_grader(BaseModel):
    """Binary Score for hallucinations present in the generative answer"""

    binary_score: str = Field(
        description="Answer is generated in the fact, 'yes' or 'no'"
    )

#LLM with function call
llm = ChatGroq(model="qwen/qwen3-32b")

structured_llm_hallucination_grader = llm.with_structured_output(hallucination_grader)

#Prompt
system = """You are a hallucination grader assessing whether the generation is grounded-in or supported by a set of retrieved facts.\n
            Give a binary score 'yes' or 'no' means that answer is grounded in or supported by the set of facts"""

hallucination_prompt = ChatPromptTemplate.from_messages(
    [
        ('system',system),
        ('human','Set of facts:\n\n {documents} \n\n LLM Generation: {generation}')
    ]
)

hallucination_grader = hallucination_prompt | structured_llm_hallucination_grader

hallucination_grader.invoke({'documents':docs,'generation':generation})

hallucination_grader(binary_score='yes')